
# FRTSearch Test Samples

This notebook evaluates FRTSearch detection instances and parameter configurations. We test against the public FRB datasets from **FAST** (<a href="https://doi.org/10.3847/1538-4365/adf42d">Guo et al. 2025</a>) and **SKA** (<a href="https://doi.org/10.1038/s41586-018-0588-y">Shannon et al. 2018</a>) analyzed in our paper.

## Dataset Overview

<table>
    <thead>
        <tr>
            <th>Source Name</th>
            <th>Telescope</th>
            <th>Reference</th>
            <th>Sampling Time (μs)</th>
            <th>Freq. Range (MHz)</th>
            <th>Channels</th>
            <th>DM (pc cm<sup>-3</sup>)</th>
            <th>ToA (s)</th>
            <th>bits</th>
        </tr>
    </thead>
    <tbody>
        <!-- FAST Group -->
        <tr>
            <td>FRB20121102</td>
            <td rowspan="3" style="vertical-align: middle; text-align: center;">FAST</td>
            <td rowspan="3" style="vertical-align: middle; text-align: center;"><a href="https://doi.org/10.3847/1538-4365/adf42d">Guo et al. (2025)</a></td>
            <td>98.304</td>
            <td>1000-1500</td>
            <td>4096</td>
            <td>565.0</td>
            <td>2.39</td>
            <td>8</td>
        </tr>
        <tr>
            <td>FRB20180301</td>
            <td>49.152</td>
            <td>1000-1500</td>
            <td>4096</td>
            <td>420.0</td>
            <td>2.83</td>
            <td>8</td>
        </tr>
        <tr>
            <td>FRB20201124</td>
            <td>49.152</td>
            <td>1000-1500</td>
            <td>4096</td>
            <td>525.0</td>
            <td>1.11</td>
            <td>8</td>
        </tr>
        <!-- SKA Group 1 -->
        <tr>
            <td>FRB20180119</td>
            <td rowspan="2" style="vertical-align: middle; text-align: center;">SKA</td>
            <td rowspan="2" style="vertical-align: middle; text-align: center;"><a href="https://doi.org/10.1038/s41586-018-0588-y">Shannon et al. (2018)</a></td>
            <td>1266.46875</td>
            <td>1130-1465</td>
            <td>336</td>
            <td>402.0</td>
            <td>1679.6</td>
            <td>8</td>
        </tr>
        <tr>
            <td>FRB20180212</td>
            <td>1266.46875</td>
            <td>1130-1465</td>
            <td>336</td>
            <td>167.0</td>
            <td>1848</td>
            <td>8</td>
        </tr>
    </tbody>
</table>

In [ ]:
# Reload modules
%matplotlib inline
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import sys
import os
import time

# Always resolve project root from this notebook file location
_nb_dir = globals().get("_dh", [os.getcwd()])[0]  # Jupyter sets _dh to notebook dir
project_root = os.path.normpath(os.path.join(_nb_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)
print(f"Working directory: {os.getcwd()}\n")

from utils import FRTDetector, PsrfitsFile, FilterbankFile, waterfall_jupyter
from mmengine.config import Config
from IPython.display import display


def detect_and_plot(file_path, config_file, slide_size=8, rank_to_display=1, 
                    freq_range=None, nsubint=None):
    """
    FRB detection and plotting with timing statistics.
    
    Args:
        file_path: Path to observation file (.fits or .fil)
        config_file: Path to detector config file
        slide_size: Sliding window size for detection
        rank_to_display: Which ranked candidate to plot (1 = best)
        freq_range: Optional (freq_min, freq_max) to override config
        nsubint: Optional number of subints for plotting to override config
    """
    times = {}
    t0 = time.time()
    
    # Stage 1: Model load and build + Data loading (combined)
    t1 = time.time()
    cfg = Config.fromfile(config_file)
    if freq_range:
        cfg.preprocess['freq_range'] = freq_range
    if nsubint is not None:
        cfg.preprocess['nsubint'] = nsubint
    detector = FRTDetector(cfg)
    detector.load_observation(file_path, slide_size=slide_size)
    times['Model load and build'] = time.time() - t1
    
    # Stage 2-4: Preprocess + Detection + IMPIC
    results = detector.get_results(coordinate_mapping=True, slide_size=slide_size)
    
    # Get accurate timing from detector
    times['Preprocess'] = detector.timing['preprocess']
    times['Detection'] = detector.timing['detection']
    times['IMPIC'] = detector.timing['impic']
    
    if results is not None and len(results) > 0:
        results_sorted = sorted(results, key=lambda x: x[2], reverse=True)
        
        print(f"\n{'='*70}")
        print(f"File: {os.path.basename(file_path)}")
        print(f"Detected {len(results_sorted)} candidates:")
        print(f"{'Rank':<6} {'TOA (s)':<12} {'DM (pc/cm³)':<15} {'Confidence':<12}")
        print(f"{'-'*70}")
        for i, (toa, dm, score) in enumerate(results_sorted, 1):
            print(f"{i:<6} {toa:<12.5f} {dm:<15.2f} {score:<12.4f}")
        print(f"{'='*70}\n")
        
        # Plot
        if rank_to_display > len(results_sorted):
            print(f"Error: Rank #{rank_to_display} exceeds {len(results_sorted)} candidates")
            return None
            
        toa, dm, score = results_sorted[rank_to_display - 1]
        print(f"Plotting Rank #{rank_to_display}: ToA={toa:.5f}s, DM={dm:.2f}, Conf={score:.4f}\n")
        
        # Stage 5: Plot
        t1 = time.time()
        preprocess = detector.preprocess
        rawdatafile = FilterbankFile(file_path) if file_path.endswith('.fil') else PsrfitsFile(file_path)
        pulsar_name = os.path.splitext(os.path.basename(file_path))[0]
        
        # Use nsubint for plotting duration
        plot_nsubint = preprocess.get('nsubint', 4)
        
        img = waterfall_jupyter(
            rawdatafile=rawdatafile, start=toa, duration=plot_nsubint,
            dm=dm, fd=preprocess.downsample_freq, downsamp=preprocess.downsample_time,
            tboxwindow=preprocess.tbox, scaling=preprocess.scaling,
            basebandStd=preprocess.basebandStd, pulsar=pulsar_name, confidence=score
        )
        times['Plot'] = time.time() - t1
        times['Total'] = time.time() - t0
        
        # Print timing summary
        print("Timing:")
        for stage in ['Model load and build', 'Preprocess', 'Detection', 'IMPIC', 'Plot', 'Total']:
            if stage in times:
                v = times[stage]
                pct = f"({v/times['Total']*100:>5.1f}%)" if stage != 'Total' else ""
                print(f"  {stage:<22}: {v:>6.2f}s  {pct}")
        print()
        
        return img
    else:
        print("No candidates detected.")
        times['Total'] = time.time() - t0
        print(f"Total time: {times['Total']:.2f}s")
        return None

# Hyperparameter Tuning Guide

**Core Constraint:** The model is trained on input dimensions of **$256 \text{ (freq)} \times 8192 \text{ (time)}$**.
*   **Standard Input:** 1 subint = 1024 time samples.
*   **Sliding Window:** Required if observation duration $> 8192$ samples.
*   **Goal:** Adjust `downsample_time`, `downsample_freq`, and `--slide-size` so the effective input dimensions match the model.

### **⚠️ General Rules**

> **🔴 CRITICAL PARAMETERS:**
> 
> 1.  **Time Dimension:** After downsampling, the window size must be **≤ 8192** ⚠️
> 
> 2.  **Frequency Dimension:** Target **≈ 256** bins. Can be higher (e.g., 336), but **⚠️ AVOID very low values** (e.g., 16) to maintain detection accuracy.
> 
> 3.  **Frequency Range:** **⚠️ ALWAYS update** `freq_range` to match your specific observation data.

## FAST-FREX FRB20121102_0038.fits
+ slide_size: 128
+ Time downsample: 16
+ Frequency downsample: 16

In [ ]:
img = detect_and_plot(
    file_path='./test_sample/FRB20121102_0038.fits',
    config_file='./configs/detector_FAST.py',
    slide_size=128,
    rank_to_display=3
)
display(img)

## FAST-FREX FRB20201124_0016.fits
+ slide_size: 128
+ Time downsample: 16
+ Frequency downsample: 16

In [ ]:
img = detect_and_plot(
    file_path='./test_sample/FRB20201124_0016.fits',
    config_file='./configs/detector_FAST.py',
    slide_size=128,
    rank_to_display=3
)
display(img)

## FAST-FREX FRB20180301_0004.fits
+ slide_size: 128
+ Time downsample: 16
+ Frequency downsample: 16

In [ ]:
img = detect_and_plot(
    file_path='./test_sample/FRB20180301_0004.fits',
    config_file='./configs/detector_FAST.py',
    slide_size=128,
    rank_to_display=2
)
display(img)

## SKA FRB2018119.fil
+ slide_size: 8
+ Time downsample: 1
+ Frequency downsample: 1

In [ ]:
img = detect_and_plot(
    file_path='./test_sample/FRB20180119_SKA_1660_1710.fil',
    config_file='./configs/detector_SKA.py',
    slide_size=8,
    rank_to_display=1
)
display(img)

## SKA FRB20180212.fil
+ slide_size: 8
+ Time downsample: 1
+ Frequency downsample: 1

In [ ]:
img = detect_and_plot(
    file_path='./test_sample/FRB20180212_SKA_1820_1870.fil',
    config_file='./configs/detector_SKA.py',
    slide_size=8,
    rank_to_display=1,
)
display(img)